# 08 · El juicio · antes vs después

**Parte III · El oficio.** En el notebook 07 entrenaste los lentes. ¿Sirvieron? "Se ve mejor" no es respuesta. Este notebook arma la tabla de la sesión 8: **mismos 10 prompts de hold-out** (fechas posteriores a todo el dataset), tres condiciones, un juez reproducible y una columna para tu juicio humano.

Condiciones:

| Condición | Qué es | ¿Entrena? |
|---|---|---|
| `zero-shot` | Qwen Instruct + system de Don Titular + notas | no |
| `few-shot` | lo mismo + **un** ejemplo resuelto del dataset antes de las notas | no |
| `lora` | system + notas, con el adapter del notebook 07 | sí |

Las tres vienen guardadas en `salidas_referencia.json`: corrida real de `Qwen2.5-0.5B-Instruct` en greedy, y un LoRA de referencia entrenado con las 44 filas de ejemplo (8 épocas, r=16; la configuración exacta está en `REF["meta"]["lora"]`). Con `GENERAR = True` reemplazas la columna `lora` por **tu** adapter. Sin torch, el notebook corre igual con lo guardado.

## Dummy (antes de la fórmula)

Un examen justo tiene tres reglas: preguntas que el alumno **no vio** (hold-out), la misma rúbrica para todos (el juez) y alguien que revise lo que la rúbrica no ve (tú).

Si la tabla dice que el LoRA mejora pero tú no publicarías ninguno de sus briefings, la tabla está midiendo otra cosa.

## Matemática (la misma idea, en serio)

Para cada condición $c$ y prompt $i\in\{1,\dots,n\}$ el juez da tres indicadores: $F_{ci}\in\{0,1\}$ (formato), $C_{ci}\in[0,1]$ (fracción de bullets bien citados) y $S_{ci}\in[0,1]$ (soporte). Reportamos medias, por ejemplo

$$
\bar F_c = \frac1n\sum_i F_{ci}, \qquad \bar S_c = \frac{1}{|I_c|}\sum_{i\in I_c} S_{ci},
$$

con $I_c$ los prompts donde hubo algo que medir. Con $n=10$ cada prompt pesa 10%: una diferencia de un caso **no** es una conclusión. La tabla sirve para ver el patrón, no para sacar un p-valor.

**Ley de Goodhart.** Si entrenas para maximizar al juez, el modelo aprende a copiar títulos (soporte 100%, formato perfecto, cero resumen). Por eso la columna humana no es decoración.

## 0 · Cargar hold-out, referencia y juez

In [ ]:
# Colab: descomenta (una vez).
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/don_titular.py
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/holdout.jsonl
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/salidas_referencia.json
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/ejemplo-briefing.jsonl

import json
import sys
from pathlib import Path

OFICIO = next((p for p in (Path("."), Path("../parte-3-oficio")) if (p / "don_titular.py").exists()), Path("."))
sys.path.insert(0, str(OFICIO.resolve()))
import don_titular as dt

HOLDOUT = [json.loads(linea) for linea in (OFICIO / "holdout.jsonl").read_text(encoding="utf-8").splitlines()]
REF = json.loads((OFICIO / "salidas_referencia.json").read_text(encoding="utf-8"))
SALIDAS: dict[str, dict[str, str]] = {k: dict(v) for k, v in REF["salidas"].items()}

print("referencia:", REF["meta"])
print("hold-out:", [c["id"] for c in HOLDOUT])

## 1 · (Opcional) Generar: tu adapter

Pon `GENERAR = True` en Colab con GPU, después de correr el gym (notebook 07). Si `ADAPTER` existe, **reemplaza** la columna `lora` de referencia por la de tu adapter. Si además pones `REGENERAR_REFERENCIA = True`, vuelve a generar zero-shot y few-shot con tu entorno (útil para comparar en la misma máquina).

In [ ]:
GENERAR = False
REGENERAR_REFERENCIA = False
MODELO = REF["meta"]["modelo"]
ADAPTER = OFICIO / "don-titular-lora"

if GENERAR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODELO)
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = AutoModelForCausalLM.from_pretrained(MODELO).to(dispositivo).eval()
    ejemplo = json.loads((OFICIO / "ejemplo-briefing.jsonl").read_text(encoding="utf-8").splitlines()[0])["messages"]

    def generar(m, mensajes):
        entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dispositivo)
        with torch.no_grad():
            out = m.generate(**entrada, max_new_tokens=REF["meta"]["max_new_tokens"], do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if REGENERAR_REFERENCIA:
        for caso in HOLDOUT:
            SALIDAS[caso["id"]]["zero-shot"] = generar(modelo, dt.mensajes(caso["notas"]))
            SALIDAS[caso["id"]]["few-shot"] = generar(modelo, ejemplo + [{"role": "user", "content": caso["notas"]}])

    if ADAPTER.exists():
        from peft import PeftModel

        con_lentes = PeftModel.from_pretrained(modelo, str(ADAPTER)).eval()
        for caso in HOLDOUT:
            SALIDAS[caso["id"]]["lora"] = generar(con_lentes, dt.mensajes(caso["notas"]))
    else:
        print(f"No hay adapter en {ADAPTER}: corre sft_lora_noticias.py primero.")

CONDICIONES = [c for c in ("zero-shot", "few-shot", "lora") if all(c in SALIDAS[k] for k in SALIDAS)]
print("condiciones:", CONDICIONES)

## 2 · La tabla

In [ ]:
JUICIOS = {(caso["id"], c): dt.juzgar(caso["notas"], SALIDAS[caso["id"]][c]) for caso in HOLDOUT for c in CONDICIONES}


def pct(x: float | None) -> str:
    return "—" if x is None else f"{x:.0%}"


def celda(j: dt.Juicio) -> str:
    return f"F{'✔' if j.formato_ok else '✘'} C{pct(j.citas):>4} S{pct(j.soporte):>4}"


print(f"{'prompt':14}" + "".join(f"{c:>18}" for c in CONDICIONES))
for caso in HOLDOUT:
    print(f"{caso['id']:14}" + "".join(f"{celda(JUICIOS[caso['id'], c]):>18}" for c in CONDICIONES))

print("\nF = 5 bullets numerados · C = % de bullets que citan un medio de las notas · S = % con soporte\n")
print(f"{'resumen':14}" + "".join(f"{c:>18}" for c in CONDICIONES))
for nombre, f in (
    ("formato", lambda js: f"{sum(j.formato_ok for j in js)}/{len(js)}"),
    ("bullets citados", lambda js: f"{sum(j.citados for j in js)}/{sum(j.con_hechos for j in js)}"),
    ("soporte medio", lambda js: pct(sum(s) / len(s)) if (s := [j.soporte for j in js if j.soporte is not None]) else "—"),
    ("problemas", lambda js: str(sum(len(j.problemas) for j in js))),
    ("0 problemas", lambda js: f"{sum(j.aprobado for j in js)}/{len(js)}"),
):
    print(f"{nombre:14}" + "".join(f"{f([JUICIOS[k, c] for k in SALIDAS]):>18}" for c in CONDICIONES))

**Cómo leer la tabla (corrida guardada).**

- **Formato: el LoRA gana sin discusión.** 10/10 contra 0/10 y 1/10. Ningún prompt logró "exactamente 5, rellena con 'no aparece'"; 44 ejemplos y un minuto de entrenamiento sí. Esto es lo que SFT enseña mejor: **forma**.
- **Citas: el LoRA no gana.** Cita bien 15 de 28 bullets: empate exacto con zero-shot, mejor que few-shot (8/28). Aprendió `Tema (Medio):` en muchos bullets, pero a veces se lo salta (`2) pemex: …`). Un hábito fino necesita más filas o más épocas.
- **Soporte: todos altos.** Con notas enfrente, Qwen casi no inventa hechos sueltos. El zero-shot es el que más adorna.
- **Problemas totales: la mitad con LoRA.** Pero "0 problemas" no es "bien": mira la sección 3.

Tu adapter tiene que ganarle a **few-shot** en lo que te importa. Si solo gana en formato, di eso, no "mejoró todo".

## 3 · Lo que el juez no vio

Lee las salidas. Estos cuatro casos están en la corrida guardada.

In [ ]:
def mostrar(caso_id: str, condicion: str) -> None:
    notas = next(c["notas"] for c in HOLDOUT if c["id"] == caso_id)
    j = JUICIOS[caso_id, condicion]
    print(f"══ {caso_id} · {condicion} · {j.resumen()}\n── notas\n{notas}\n── salida\n{SALIDAS[caso_id][condicion]}\n")


for caso_id, condicion in (("h1-economia", "zero-shot"), ("h4-movilidad", "few-shot"), ("h10-mixto", "lora"), ("h3-deportes", "lora")):
    mostrar(caso_id, condicion)

- **h1 · zero-shot.** "…subió a 4.1%, lo que es menos que la prevista": las notas no dicen nada de lo previsto. De las 6 raíces del renglón, 3 están en las notas (`infla`, `prime`, `quinc`): soporte exactamente 50%, el umbral, y pasa. El juez sí cachó el medio inventado ("La República") y "Economía" usado como si fuera un medio.
- **h4 · few-shot.** Soporte 100%, y aun así está mal: le atribuye el cierre por la marcha a **Metro CDMX** (era *Local*) y convierte la avenida **Reforma** en un medio. Las palabras están en las notas; las relaciones no. Un juez de bolsa de palabras no ve relaciones.
- **h10 · lora.** **Cero problemas** para el juez, y está mal: "Viaducto (ESPN): cierre parcial" (el cierre lo reportó *Local*, ESPN dio el futbol) y "Olimpíada (Local): viaducto" no significa nada. Cada medio existe en las notas y cada palabra también. Formato perfecto, atribución rota: esto es Goodhart en vivo.
- **h3 · lora.** "5) Deportes: no aparece" en un día con **tres** notas de deportes. El LoRA aprendió que los últimos bullets se rellenan con "no aparece" y lo aplica aunque mienta. El juez no revisa si un "no aparece" es cierto.

## 4 · Tu columna: ¿lo publicarías?

Llena 1 (sí, tal cual), 0.5 (con una corrección) o 0 (no). Es la columna que más pesa: el juez no lee, tú sí.

In [ ]:
HUMANO: dict[tuple[str, str], float | None] = {(k, c): None for k in SALIDAS for c in CONDICIONES}
# Ejemplo: HUMANO["h10-mixto", "lora"] = 0   # atribución rota aunque el juez diga 0 problemas

for c in CONDICIONES:
    notas = [v for (k, cc), v in HUMANO.items() if cc == c and v is not None]
    print(f"{c:10} · humano: " + (f"{sum(notas) / len(notas):.2f} ({len(notas)}/{len(SALIDAS)} revisados)" if notas else "sin revisar"))

---

## Entregable (sesión 8)

1. La tabla de la sección 2 con **tres** columnas (`zero-shot`, `few-shot`, `lora` con **tu** adapter) en **tu** hold-out de ≥ 5 prompts, separado por fecha (notebook 06).
2. La columna humana llena, y un caso donde el juez y tú no estén de acuerdo.
3. Una frase honesta: ¿tu LoRA le gana a few-shot? Si no, ¿por qué crees (datos, épocas, r)?

## Para romperlo

- Agrega a `don_titular.juzgar` una regla de **atribución**: el medio citado en un bullet debe ser el de la nota con mayor soporte para ese bullet. ¿Caza h10 · lora?
- Agrega una regla de **"no aparece" honesto**: si el tema de un bullet "no aparece" comparte raíces con alguna nota, es omisión. ¿Caza h3 · lora?
- Haz few-shot con **tres** ejemplos. ¿Se acerca al formato? ¿Cuántos tokens extra cuesta por llamada?
- Entrena un LoRA con un dataset donde **todas** las filas copien títulos. Mira cómo sube el juez y baja tu columna. Eso es Goodhart.